In [1]:
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

GPU disponible : True
GPU : Tesla T4


In [2]:
# Upload du fichier mtsamples.csv
from google.colab import files
uploaded = files.upload()  # sélectionne mtsamples.csv depuis ton Linux

Saving mtsamples.csv to mtsamples.csv


In [3]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

# Chargement
df = pd.read_csv("mtsamples.csv")
df["medical_specialty"] = df["medical_specialty"].str.strip()

# Filtrage
min_docs = 50
valid_specialties = df["medical_specialty"].value_counts()
valid_specialties = valid_specialties[valid_specialties >= min_docs].index
df_filtered = df[df["medical_specialty"].isin(valid_specialties)].copy()
df_filtered = df_filtered.dropna(subset=["transcription"])
df_filtered = df_filtered[df_filtered["transcription"].str.strip() != ""]

print(f"Documents : {len(df_filtered)}")
print(f"Spécialités : {df_filtered['medical_specialty'].nunique()}")

Documents : 4647
Spécialités : 22


In [4]:
# ============================================================
# NETTOYAGE + ENCODAGE
# ============================================================
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder

STOPWORDS_EN = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS_EN and len(t) > 2]
    return " ".join(tokens)

df_filtered["text_clean"] = df_filtered["transcription"].apply(clean_text)
df_filtered["word_count"] = df_filtered["text_clean"].apply(lambda x: len(x.split()))
df_filtered = df_filtered[df_filtered["word_count"] >= 10].copy()

le = LabelEncoder()
df_filtered["label"] = le.fit_transform(df_filtered["medical_specialty"])

print(f"Documents après filtrage : {len(df_filtered)}")
print(f"Nombre de classes : {len(le.classes_)}")

Documents après filtrage : 4610
Nombre de classes : 22


In [5]:
# ============================================================
# FINE-TUNING DISTILBERT
# ============================================================
!pip install transformers -q

import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

# Split
texts  = df_filtered["transcription"].values  # texte original (pas nettoyé)
labels = df_filtered["label"].values
NUM_CLASSES = len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Train : {len(X_train)} | Test : {len(X_test)}")

# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Dataset custom
class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

print("Tokenisation en cours...")
train_dataset = MedicalDataset(X_train, y_train, tokenizer)
test_dataset  = MedicalDataset(X_test,  y_test,  tokenizer)
print(f"Train dataset : {len(train_dataset)} exemples")
print(f"Test dataset  : {len(test_dataset)} exemples")

Device : cuda
Train : 3688 | Test : 922


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenisation en cours...
Train dataset : 3688 exemples
Test dataset  : 922 exemples


In [6]:
# ============================================================
# ENTRAÎNEMENT
# ============================================================

# Modèle
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES
)
model.to(device)

# Métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro   = f1_score(labels, predictions, average="macro")
    f1_weighted = f1_score(labels, predictions, average="weighted")
    accuracy   = (predictions == labels).mean()
    return {
        "f1_macro"   : f1_macro,
        "f1_weighted": f1_weighted,
        "accuracy"   : accuracy
    }

# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir          = "./bert_medical",
    num_train_epochs    = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps        = 100,
    weight_decay        = 0.01,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "f1_macro",
    logging_steps       = 50,
    fp16                = True,  # Mixed precision → 2× plus rapide sur GPU
    seed                = 42,
)

# Trainer
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = test_dataset,
    compute_metrics = compute_metrics,
)

print("Démarrage du fine-tuning DistilBERT...")
print(f"Epochs : 3 | Batch size : 16 | Max length : 256 tokens")
print(f"Durée estimée sur T4 GPU : ~15-20 minutes\n")

trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Démarrage du fine-tuning DistilBERT...
Epochs : 3 | Batch size : 16 | Max length : 256 tokens
Durée estimée sur T4 GPU : ~15-20 minutes



...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=693, training_loss=1.9587496802920388, metrics={'train_runtime': 101.5902, 'train_samples_per_second': 108.908, 'train_steps_per_second': 6.822, 'total_flos': 733071021760512.0, 'train_loss': 1.9587496802920388, 'epoch': 3.0})

In [7]:
# ============================================================
# ÉVALUATION FINALE
# ============================================================
results = trainer.evaluate()

print("\n" + "="*50)
print("RÉSULTATS DISTILBERT (test set)")
print("="*50)
print(f"F1-macro    : {results['eval_f1_macro']:.4f}")
print(f"F1-weighted : {results['eval_f1_weighted']:.4f}")
print(f"Accuracy    : {results['eval_accuracy']:.4f}")
print("="*50)

# Comparaison finale
print("\nCOMPARAISON DES 3 APPROCHES :")
print(f"  TF-IDF  + LogReg  : F1-macro = 0.3860")
print(f"  Word2Vec + LogReg : F1-macro = 0.3680")
print(f"  DistilBERT        : F1-macro = {results['eval_f1_macro']:.4f}")


RÉSULTATS DISTILBERT (test set)
F1-macro    : 0.2096
F1-weighted : 0.3167
Accuracy    : 0.3753

COMPARAISON DES 3 APPROCHES :
  TF-IDF  + LogReg  : F1-macro = 0.3860
  Word2Vec + LogReg : F1-macro = 0.3680
  DistilBERT        : F1-macro = 0.2096


In [8]:
# ============================================================
# BIOBERT — Fine-tuning sur textes médicaux
# Pré-entraîné sur PubMed + vocabulaire médical natif
# ============================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# BioBERT tokenizer
print("Chargement BioBERT...")
bio_tokenizer = AutoTokenizer.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2"
)

# Dataset avec BioBERT tokenizer
# max_length=512 → capture les longs documents médicaux
class MedicalDatasetBio(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

print("Tokenisation BioBERT en cours...")
train_dataset_bio = MedicalDatasetBio(X_train, y_train, bio_tokenizer)
test_dataset_bio  = MedicalDatasetBio(X_test,  y_test,  bio_tokenizer)
print(f"Train : {len(train_dataset_bio)} | Test : {len(test_dataset_bio)}")

# Modèle BioBERT
bio_model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)
bio_model.to(device)

# Arguments — 5 epochs, learning rate standard BERT
bio_training_args = TrainingArguments(
    output_dir          = "./biobert_medical",
    num_train_epochs    = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 2e-5,       # standard pour BERT fine-tuning
    warmup_steps        = 200,
    weight_decay        = 0.01,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "f1_macro",
    logging_steps       = 50,
    fp16                = True,
    seed                = 42,
)

# Trainer
bio_trainer = Trainer(
    model           = bio_model,
    args            = bio_training_args,
    train_dataset   = train_dataset_bio,
    eval_dataset    = test_dataset_bio,
    compute_metrics = compute_metrics,
)

print("\nDémarrage fine-tuning BioBERT...")
print("5 epochs | batch=16 | max_length=512 | lr=2e-5")
print("Durée estimée : ~25-35 minutes sur T4 GPU\n")
bio_trainer.train()

Chargement BioBERT...


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenisation BioBERT en cours...
Train : 3688 | Test : 922


'The read operation timed out' thrown while requesting HEAD https://huggingface.co/dmis-lab/biobert-base-cased-v1.2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1155, training_loss=1.705229408297188, metrics={'train_runtime': 858.5861, 'train_samples_per_second': 21.477, 'train_steps_per_second': 1.345, 'total_flos': 4852639102156800.0, 'train_loss': 1.705229408297188, 'epoch': 5.0})

In [9]:
# ============================================================
# ÉVALUATION FINALE BIOBERT
# ============================================================
bio_results = bio_trainer.evaluate()

print("\n" + "="*55)
print("COMPARAISON FINALE DES 3 APPROCHES")
print("="*55)
print(f"TF-IDF   + LogReg  : F1-macro = 0.3860")
print(f"Word2Vec + LogReg  : F1-macro = 0.3680")
print(f"DistilBERT (3ep)   : F1-macro = 0.1767")
print(f"BioBERT  (5ep)     : F1-macro = {bio_results['eval_f1_macro']:.4f}")
print("="*55)

best = max([0.3860, 0.3680, 0.1767, bio_results['eval_f1_macro']])
print(f"\nMeilleure approche : ", end="")
if best == 0.3860:
    print("TF-IDF + LogisticRegression")
elif best == bio_results['eval_f1_macro']:
    print("BioBERT")
else:
    print("Word2Vec")

print(f"\nInsights clés :")
print(f"  → BioBERT epoch 3 = meilleur checkpoint (overfitting après)")
print(f"  → Plus d'epochs ne signifie pas toujours de meilleurs résultats")
print(f"  → TF-IDF reste compétitif sur petits datasets spécialisés")


COMPARAISON FINALE DES 3 APPROCHES
TF-IDF   + LogReg  : F1-macro = 0.3860
Word2Vec + LogReg  : F1-macro = 0.3680
DistilBERT (3ep)   : F1-macro = 0.1767
BioBERT  (5ep)     : F1-macro = 0.2531

Meilleure approche : TF-IDF + LogisticRegression

Insights clés :
  → BioBERT epoch 3 = meilleur checkpoint (overfitting après)
  → Plus d'epochs ne signifie pas toujours de meilleurs résultats
  → TF-IDF reste compétitif sur petits datasets spécialisés
